# Cacifer QA: Fine-tuning DistilBERT on a Literary Novel

In this notebook I fine-tune an extractive QA model (`distilbert-base-uncased-distilled-squad`) on my own novel *Cacifer 1264 – Book One (Monastery)*.  
I load the full text from a Kaggle dataset, design story-aware question–answer pairs, and automatically convert them into SQuAD-style training examples.  
Then I fine-tune the model on these examples and test how well it can answer new factual questions about Romé, Dania, Deonardo, and the monastery setting.  
The goal is to practice Hugging Face–style QA fine-tuning on a high-level literary text, not to build a production system.


# Step 1 – Load base QA model and CAcifer's novel chunk 

In [1]:
!pip install -q transformers datasets accelerate


In [2]:
from transformers import pipeline

qa_pipeline = pipeline(
    task="question-answering",
    model="distilbert-base-uncased-distilled-squad"
)

with open("/kaggle/input/datasets/asopozala/cacifer1264-bookone/Cacifer1264_BookOne_Monastry.txt",
          "r", encoding="utf-8") as f:
    novel_text = f.read()


config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
question = "How did Romé first arrive at San Michele?"
result = qa_pipeline(question=question, context=novel_text[:4000])  # first chunk as a test
print(result)


{'score': 0.15482475562021136, 'start': 2651, 'end': 2685, 'answer': 'exiled the bandits and laid stones'}


In [4]:
arrival_phrase = "That was how Romé came to San Michele"
start = novel_text.index(arrival_phrase) - 400   # a bit before
end = start + 800                                # a bit after
context = novel_text[start:end]

print(context)  # just to check the slice

result = qa_pipeline(
    question="How did Romé first arrive at San Michele?",
    context=context
)
print(result)


ho rides here at this hour, in such weather? 
By God, no pilgrim could come so — not with those shoulders, not with that bearing… wounded, yes, yet not broken. Look at him. 
Look at those eyes… I cannot…


The thought stumbled into silence, shocked by its own astonishment. I felt the man’s beauty strike the gatekeeper like cold water: unbelievable, unreasonable, out of place in frost and night.


That was how Romé came to San Michele — first seen not by my eyes, but in the trembling mind of the one who opened the gate.


The Fire and the First Silence 🕯♛♡🐾 cacifer’s book




I had dropped from the falcon, returned my body of a cat to the monastery.


The gate groaned open, wood swollen with frost. The red horse’s breath smoked into the cloister square, hooves striking hard against stone. S
{'score': 0.21803653240203857, 'start': 471, 'end': 523, 'answer': 'in the trembling mind of the one who opened the gate'}


In [5]:
snippet_start = novel_text.index("A man on horseback")
context_start = max(0, snippet_start - 300)
context_end   = snippet_start + 300
context = novel_text[context_start:context_end]

print(context)  # check that it contains the sentence


he path was only a pale vein through darkness, the gate of San Michele half-buried.


I watched the gatekeeper half-run, half-walk across the square, breath steaming, torn between duty and wonder. Snow scattered beneath the horse’s hooves, the first sound to break the valley’s hush in three days.


A man on horseback, cloaked, wounded, snow crusted into his mantle. I could not see his face yet. But the gatekeeper saw him, and I heard his mind flare open —


Ah,What is this? Who rides here at this hour, in such weather? 
By God, no pilgrim could come so — not with those shoulders, not with that


In [6]:
answer_text = "A man on horseback, cloaked, wounded, snow crusted into his mantle."
answer_start = context.index(answer_text)  # now inside this new context

print(answer_start)


300


In [7]:
from datasets import Dataset

answer_text = "A man on horseback, cloaked, wounded, snow crusted into his mantle."

train_examples = [{
    "id": "qa1",
    "title": "cacifer",
    "context": context,
    "question": "How did Romé first arrive at San Michele?",
    "answers": {
        "text": [answer_text],
        "answer_start": [300],
    },
}]

train_dataset = Dataset.from_list(train_examples)
train_dataset, train_dataset[0]["answers"]


(Dataset({
     features: ['id', 'title', 'context', 'question', 'answers'],
     num_rows: 1
 }),
 {'answer_start': [300],
  'text': ['A man on horseback, cloaked, wounded, snow crusted into his mantle.']})

In [8]:
from datasets import Dataset

specs = [
    {
        "id": "qa1",
        "question": "How did Romé first arrive at San Michele?",
        "answer_text": "A man on horseback, cloaked, wounded, snow crusted into his mantle.",
    },
    {
        "id": "qa2",
        "question": "What is Deonardo's full name?",
        "answer_text": "Deonardo di Alferini.",
    },
    {
        "id": "qa3",
        "question": "Who requested the old herbals that brought Deonardo to San Michele?",
        "answer_text": "Donna Aurelia di Vercado’s request for those old herbals",
    },
    {
        "id": "qa4",
        "question": "Where is Dania's healer's garden?",
        "answer_text": "Her terrace opened on the slope below, just before the ridge dropped sheer into the cliff.",
    },
]

examples = []
for spec in specs:
    pos = novel_text.index(spec["answer_text"])
    ctx_start = max(0, pos - 400)
    ctx_end = pos + 400
    ctx = novel_text[ctx_start:ctx_end]
    answer_start = ctx.index(spec["answer_text"])
    examples.append({
        "id": spec["id"],
        "title": "cacifer",
        "context": ctx,
        "question": spec["question"],
        "answers": {"text": [spec["answer_text"]], "answer_start": [answer_start]},
    })

train_dataset = Dataset.from_list(examples)
train_dataset


Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 4
})

# 1. Load the base QA model and tokenizer

In [9]:
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)

model_name = "distilbert-base-uncased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

# 2. Convert the dataset to model features (adds start/end token positions)

In [10]:
def preprocess_qa(example):
    tokenized = tokenizer(
        example["question"],
        example["context"],
        truncation="only_second",
        max_length=512,
        padding="max_length",
        return_offsets_mapping=True,
    )

    offsets = tokenized["offset_mapping"]
    start_char = example["answers"]["answer_start"][0]
    end_char = start_char + len(example["answers"]["text"][0])
    sequence_ids = tokenized.sequence_ids()

    # find context token range
    context_start = next(i for i, sid in enumerate(sequence_ids) if sid == 1)
    context_end = max(i for i, sid in enumerate(sequence_ids) if sid == 1)

    start_pos = end_pos = None
    for i in range(context_start, context_end + 1):
        if offsets[i][0] <= start_char < offsets[i][1]:
            start_pos = i
        if offsets[i][0] < end_char <= offsets[i][1]:
            end_pos = i
            break

    tokenized["start_positions"] = start_pos
    tokenized["end_positions"] = end_pos
    tokenized.pop("offset_mapping")
    return tokenized

tokenized_train = train_dataset.map(preprocess_qa)


Map:   0%|          | 0/4 [00:00<?, ? examples/s]


# 3. Fine‑tune briefly and test on a question

In [11]:
args = TrainingArguments(
    output_dir="./cacifer-qa",
    per_device_train_batch_size=2,
    learning_rate=3e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=1,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
)

trainer.train()

from transformers import pipeline
finetuned_qa = pipeline("question-answering", model=model, tokenizer=tokenizer)
print(finetuned_qa(
    question="How did Romé first arrive at San Michele?",
    context=tokenized_train[0]["context"] if "context" in tokenized_train[0] else train_dataset[0]["context"],
))


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,3.603345
2,3.356853
3,2.444381
4,0.956335
5,0.946772
6,0.748268
7,0.665184
8,0.450171
9,0.441895
10,0.385085


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'score': 0.033864010125398636, 'start': 400, 'end': 442, 'answer': 'A man on horseback, cloaked, wounded, snow'}
